In [0]:
%sql
WITH calendario AS (
    SELECT 
        explode(
            sequence(
                (SELECT MIN(sale_date) FROM fatos_vendas),
                (SELECT MAX(sale_date) FROM fatos_vendas),
                interval 1 day
            )
        ) AS data
),
vendas_por_dia AS (
    SELECT 
        sale_date,
        SUM(total) AS receita_dia
    FROM fatos_vendas
    GROUP BY sale_date
),
base AS (
    SELECT 
        c.data,
        COALESCE(v.receita_dia, 0) AS receita_dia
    FROM calendario c
    LEFT JOIN vendas_por_dia v
        ON c.data = v.sale_date
)
SELECT 
    dayofweek(data) AS dia_semana_num,

    CASE dayofweek(data)
        WHEN 1 THEN 'Domingo'
        WHEN 2 THEN 'Segunda-feira'
        WHEN 3 THEN 'Terça-feira'
        WHEN 4 THEN 'Quarta-feira'
        WHEN 5 THEN 'Quinta-feira'
        WHEN 6 THEN 'Sexta-feira'
        WHEN 7 THEN 'Sábado'
    END AS dia_semana,

    ROUND(AVG(receita_dia), 2) AS media_vendas
FROM base
GROUP BY dia_semana_num, dia_semana
ORDER BY dia_semana_num;